# Lesson 09: FoodVision Big - Production-Ready Food Classifier

This notebook builds a **production-scale food classifier** using:
- **Food101 dataset**: 101 food categories, 1000 images each
- **EfficientNet-B2**: Larger pretrained model for better accuracy
- **20% of data**: Still 15,150 training images (101 classes × 750 images × 0.2)
- **Transfer learning**: Feature extraction approach
- **Data augmentation**: TrivialAugmentWide for robustness

## Goal:
Create a model that can classify 101 different food dishes with high accuracy

## Why This Matters:
This is a **real-world scale** problem:
- Many classes (101 vs 3)
- Large dataset (75,000+ images)
- Practical application (food recognition apps)
- Production considerations (model size, speed, accuracy)

## Key Techniques:
1. **Using torchvision.datasets.Food101** - built-in dataset
2. **Data splitting** - using subset for faster experimentation
3. **Label smoothing** - regularization technique
4. **TrivialAugmentWide** - automatic strong augmentation
5. **Model evaluation** - on 101-class problem

## Setup and Imports

In [ ]:
# Check Python version
import sys
sys.version

In [ ]:
# Check PyTorch and CUDA setup
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

# For this notebook, GPU is highly recommended!
# Training 101 classes on CPU would take hours

In [ ]:
# Import standard libraries
import torchvision
from torch import nn
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import time  # For timing experiments

In [ ]:
# Try to get torchinfo and going_modular
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Import our modular code from previous lessons
try:
    from going_modular import data_setup, engine
    from going_modular.utils import download_data
except:
    print("[INFO] Couldn't find going_modular... downloading from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular import data_setup, engine
    from going_modular.utils import download_data

In [ ]:
# Setup device
# For Food101, GPU makes a HUGE difference
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Optional: Download 20% Subset for Testing

Before working with full Food101, you can test with a smaller subset.

In [ ]:
# Download 20% pizza/steak/sushi data for initial testing
# This is useful for debugging before scaling to Food101
data_20_percent_path = download_data(
    source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
    destination="pizza_steak_sushi_20_percent"
)

data_20_percent_path

In [ ]:
# Setup directory paths
train_dir = data_20_percent_path / "train"
test_dir = data_20_percent_path / "test"

## Part 1: Create EfficientNet-B2 Model

### Why EfficientNet-B2?
- **Better than B0**: More parameters, better accuracy
- **Not too big**: B2 is still reasonable for fine-tuning
- **Scalable**: Part of a family (B0-B7)
- **Efficient**: Good accuracy-to-parameters ratio

### Model Statistics:
- **Parameters**: ~9M (vs ~5M for B0)
- **Image size**: 260×260 (vs 224×224 for B0)
- **Top-1 Accuracy on ImageNet**: ~80.5%

In [ ]:
def create_effnetb2_model(num_classes: int=3, seed: int=42):
    """
    Creates an EfficientNetB2 feature extractor model and transforms.

    Args:
        num_classes (int, optional): Number of classes in the classifier head. 
            Defaults to 3.
        seed (int, optional): Random seed value for reproducibility. 
            Defaults to 42.

    Returns:
        model (torch.nn.Module): EfficientNetB2 feature extractor model.
        transforms (torchvision.transforms): EfficientNetB2 image transforms.
    """
    # Get pretrained weights for EfficientNet-B2
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    
    # Get transforms used for pretraining
    # CRITICAL: Use same preprocessing as during pretraining!
    transforms = weights.transforms()
    
    # Create model with pretrained weights
    model = torchvision.models.efficientnet_b2(weights=weights)

    # Freeze all base layers (feature extractor)
    # We only train the classifier head
    for param in model.features.parameters():
        param.requires_grad = False

    # Set seed for reproducibility
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
    # Replace classifier head for our number of classes
    # EfficientNet-B2 outputs 1408 features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),  # Regularization
        nn.Linear(
            in_features=1408,  # B2 feature dimension
            out_features=num_classes  # Our classes
        )
    )
    
    return model, transforms

In [ ]:
# Create EfficientNet-B2 model for Food101 (101 classes)
# Use current time as seed for variability across runs
effnetb2_food101, effnetb2_food101_transforms = create_effnetb2_model(
    num_classes=101,
    seed=int(time.time())
)

# Move model to GPU
effnetb2_food101 = effnetb2_food101.to(device)

In [ ]:
# Print model summary
# Notice: Most parameters are NOT trainable (frozen)
summary(
    effnetb2_food101,
    input_size=(1, 3, 224, 224),
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

# Key observations:
# - Total params: ~9M
# - Trainable params: ~145K (just the classifier!)
# - Non-trainable: ~8.9M (frozen features)

## Part 2: Setup Data Augmentation

### TrivialAugmentWide:
- **Automatic augmentation** strategy
- **Randomly applies** one transformation per image
- **Wide variety**: rotation, translation, color, shear, etc.
- **Research-backed**: From "TrivialAugment" paper
- **Easy to use**: No hyperparameter tuning needed

### Why Augmentation?
1. **Prevent overfitting**: Model sees varied data
2. **Better generalization**: Works on diverse real-world images
3. **Data efficiency**: Get more from limited data

In [ ]:
# Create Food101 training transforms
# Combine TrivialAugmentWide with pretrained model transforms

food101_train_transforms = torchvision.transforms.Compose([
    # Apply strong augmentation first
    torchvision.transforms.TrivialAugmentWide(),
    # Then apply model's expected preprocessing
    effnetb2_food101_transforms
])

print(f"Training transforms: {food101_train_transforms}")

# Note: Test transforms don't need augmentation
# Just use effnetb2_food101_transforms for testing

## Part 3: Load Food101 Dataset

### Food101 Dataset:
- **101 food categories** (apple pie, baby back ribs, pizza, etc.)
- **1000 images per class** (101,000 total images)
- **750 training images** per class (75,750 total)
- **250 test images** per class (25,250 total)
- **Challenging**: Real-world images with variation

### Built into torchvision!
- `torchvision.datasets.Food101` downloads automatically
- No manual downloading needed
- Standard PyTorch Dataset interface

In [ ]:
from torchvision import datasets

In [ ]:
# Setup data directory
from pathlib import Path
data_dir = Path("data")

# Get training data (~750 images × 101 food classes = 75,750 images)
# First run will download the dataset (~5GB)
train_data = datasets.Food101(
    root=data_dir,  # Where to download/store data
    split="train",  # Training split
    transform=food101_train_transforms,  # Apply augmentation
    download=True  # Download if not already present
)

# Get test data (~250 images × 101 classes = 25,250 images)
test_data = datasets.Food101(
    root=data_dir,
    split="test",  # Test split
    transform=effnetb2_food101_transforms,  # No augmentation for test
    download=True
)

print(f"Training samples: {len(train_data):,}")
print(f"Testing samples: {len(test_data):,}")
print(f"Classes: {len(train_data.classes)}")

## Part 4: Create Data Subsets (20% for Faster Experimentation)

### Why use subsets?
- **Faster iteration**: 20% = ~15K images vs 75K
- **Quick experiments**: Test ideas before full training
- **Resource efficient**: Less GPU memory, faster epochs
- **Good practice**: Validate approach before scaling

### Trade-off:
- Less data = slightly lower accuracy
- But much faster development cycle!

In [ ]:
def split_dataset(
    dataset: torchvision.datasets,
    split_size: float=0.2,
    seed: int=42
):
    """
    Randomly splits a dataset into two proportions based on split_size and seed.

    Args:
        dataset (torchvision.datasets): A PyTorch Dataset.
        split_size (float, optional): Proportion for the first split. 
            Defaults to 0.2 (20%).
        seed (int, optional): Random seed for reproducibility. Defaults to 42.

    Returns:
        tuple: (subset_1, subset_2) as torch.utils.data.Subset objects.
        
    Example:
        # Create 20% and 80% splits
        train_20_percent, train_80_percent = split_dataset(train_data, split_size=0.2)
    """
    # Calculate split lengths
    length_1 = int(len(dataset) * split_size)
    length_2 = len(dataset) - length_1
    
    print(f"[INFO] Splitting dataset of length {len(dataset)} into splits of size: {length_1} ({split_size*100:.0f}%), {length_2} ({(1-split_size)*100:.0f}%)")
    
    # Use random_split with generator for reproducibility
    subset_1, subset_2 = torch.utils.data.random_split(
        dataset,
        lengths=[length_1, length_2],
        generator=torch.manual_seed(seed)  # Reproducible splits
    )
    
    return subset_1, subset_2

In [ ]:
# Create 20% training subset
# 75,750 × 0.2 = 15,150 images
train_data_food101_20_percent, _ = split_dataset(
    dataset=train_data,
    split_size=0.2
)

# Create 20% testing subset  
# 25,250 × 0.2 = 5,050 images
test_data_food101_20_percent, _ = split_dataset(
    dataset=test_data,
    split_size=0.2
)

print(f"\n20% training samples: {len(train_data_food101_20_percent):,}")
print(f"20% testing samples: {len(test_data_food101_20_percent):,}")

## Part 5: Create DataLoaders

In [ ]:
import os

In [ ]:
# Setup DataLoader hyperparameters
BATCH_SIZE = 32  # Process 32 images at a time

# Set num_workers based on CPU count
# More workers = faster data loading (up to a point)
NUM_WORKERS = 2 if os.cpu_count() <= 4 else 4

print(f"Using {NUM_WORKERS} workers for data loading")

# Create training DataLoader
train_dataloader_food101_20_percent = torch.utils.data.DataLoader(
    train_data_food101_20_percent,
    batch_size=BATCH_SIZE,
    shuffle=True,  # Shuffle for better training
    num_workers=NUM_WORKERS,
    pin_memory=True  # Faster GPU transfer
)

# Create testing DataLoader
test_dataloader_food101_20_percent = torch.utils.data.DataLoader(
    test_data_food101_20_percent,
    batch_size=BATCH_SIZE,
    shuffle=False,  # No need to shuffle test data
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"\nTraining batches: {len(train_dataloader_food101_20_percent)}")
print(f"Testing batches: {len(test_dataloader_food101_20_percent)}")

## Part 6: Train the Model

### Training Configuration:
- **Optimizer**: Adam with lr=1e-3
- **Loss**: CrossEntropyLoss with label smoothing
- **Epochs**: 5-10 (can increase for better results)

### Label Smoothing:
- **What**: Softens hard labels (0/1) to (ε/K, 1-ε+ε/K)
- **Why**: Prevents overconfidence, better generalization
- **Value**: 0.1 is common (10% smoothing)

### Expected Performance:
- **20% data**: ~50-60% accuracy on 101 classes
- **100% data**: ~75-80% accuracy possible
- **Training time**: ~5-10 min per epoch on GPU

In [ ]:
from going_modular import engine

In [ ]:
# Uncomment to train the model
# WARNING: This will take significant time!

# # Setup optimizer
# optimizer = torch.optim.Adam(
#     params=effnetb2_food101.parameters(),
#     lr=1e-3  # Learning rate
# )

# # Setup loss function with label smoothing
# # Label smoothing helps prevent overconfidence
# loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

# # Set seeds for reproducibility
# torch.manual_seed(42)
# torch.cuda.manual_seed(42)

# # Train the model
# effnetb2_food101_results = engine.train(
#     model=effnetb2_food101,
#     train_dataloader=train_dataloader_food101_20_percent,
#     test_dataloader=test_dataloader_food101_20_percent,
#     optimizer=optimizer,
#     loss_fn=loss_fn,
#     epochs=5,  # Can increase for better results
#     device=device
# )

In [ ]:
# Uncomment to plot training curves
# from helper_functions import plot_loss_curves

# # Visualize training progress
# plot_loss_curves(effnetb2_food101_results)

## Part 7: Save and Load Model

In [ ]:
from going_modular import utils

# Create model save path
effnetb2_food101_model_path = "09_pretrained_effnetb2_feature_extractor_food101_20_percent.pth"

# Uncomment to save trained model
# utils.save_model(
#     model=effnetb2_food101,
#     target_dir="models",
#     model_name=effnetb2_food101_model_path
# )

In [ ]:
# Load a pretrained model
# Create fresh model instance
loaded_effnetb2_food101, effnetb2_transforms = create_effnetb2_model(num_classes=101)

# Load saved weights
loaded_effnetb2_food101.load_state_dict(
    torch.load(f"models/{effnetb2_food101_model_path}")
)

# Move to device
loaded_effnetb2_food101 = loaded_effnetb2_food101.to(device)

print(f"[INFO] Loaded model from: models/{effnetb2_food101_model_path}")

In [ ]:
# Check model file size
from pathlib import Path

# Get size in bytes, convert to MB
pretrained_effnetb2_food101_model_size = (
    Path("models", effnetb2_food101_model_path).stat().st_size // (1024*1024)
)

print(f"Model size: {pretrained_effnetb2_food101_model_size} MB")

# Typical size: ~30-35 MB
# Larger than 3-class model due to 101 output classes

## Part 8: Make Predictions on Custom Images

In [ ]:
# Setup custom image path (update with your image path)
custom_image_path = r"path/to/your/food/image.jpg"

# Load Food101 class names
# These are the 101 food categories
meta_data_path = r"path/to/food-101/meta/classes.txt"
class_names = np.loadtxt(meta_data_path, dtype=str)

print(f"Number of classes: {len(class_names)}")
print(f"Sample classes: {class_names[:5]}")

In [ ]:
from going_modular.predictions import pred_and_plot_image

# Make prediction on custom image
pred_and_plot_image(
    model=loaded_effnetb2_food101,
    image_path=custom_image_path,
    class_names=class_names,
    transform=effnetb2_transforms,
    device=device
)

# The model will predict one of 101 food classes
# and show the confidence (probability)

## Summary: Building Production-Scale Food Classifiers

### What We Built:
- **101-class food classifier** using EfficientNet-B2
- **Transfer learning** with feature extraction
- **Data augmentation** with TrivialAugmentWide
- **Efficient training** on 20% subset

### Key Learnings:

**1. Scaling to More Classes**
- 101 classes vs 3: Much harder!
- Need more data per class
- Stronger augmentation helps
- Bigger models (B2 vs B0) help

**2. Working with Built-in Datasets**
- `torchvision.datasets.Food101` convenience
- Automatic downloading
- Standard splits (train/test)
- Easy integration

**3. Data Efficiency**
- 20% subset for quick iteration
- Still ~15K images = good results
- Full dataset for production

**4. Production Considerations**
- **Model size**: ~30 MB (deployable)
- **Inference speed**: Real-time on GPU
- **Accuracy**: 50-60% on 20%, 75-80% on 100%
- **Memory**: Manageable with batching

### Improving the Model:

**More Data:**
- Use 100% of Food101 (75K images)
- Better: ~75-80% accuracy

**Better Models:**
- Try EfficientNet-B3, B4 (more parameters)
- Try Vision Transformer (ViT)
- Ensemble multiple models

**Better Training:**
- Train longer (20-50 epochs)
- Use learning rate scheduling
- Try different optimizers (AdamW, SGD)
- Fine-tune some layers (unfreeze gradually)

**Better Augmentation:**
- Adjust TrivialAugmentWide strength
- Add test-time augmentation
- Use CutMix/MixUp

### Real-World Applications:
- **Food tracking apps**: Count calories automatically
- **Restaurant menus**: Identify dishes
- **Nutrition apps**: Dietary recommendations
- **Social media**: Auto-tag food photos
- **Cooking assistants**: Recipe suggestions

### Next Steps:
1. Train on full Food101 dataset
2. Experiment with larger models (B3, B4)
3. Try fine-tuning (unfreeze some layers)
4. Deploy as a web app or mobile app
5. Collect your own food images and test
6. Consider model compression for mobile